# 02.01 — Pretrained screening evidence

Câu hỏi: ba candidate có khác nhau thế nào trên cùng validation? Notebook đọc đủ 12 run của OPUS-MT, EnViT5 và NLLB-200; bảng chỉ hiển thị model, chiều dịch, dataset, số câu và hai metric. Thông tin provenance ở bên dưới; không có bảng nào tự chọn candidate.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'evidence/phase02').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Cannot locate the project root containing evidence/phase02.')
BASE = ROOT / 'evidence/phase02'
rows = []
for metric_path in BASE.glob('*/*/pretrained/*/metrics.json'):
    run_path = metric_path.with_name('run.json')
    if not run_path.is_file():
        continue
    metrics = json.loads(metric_path.read_text(encoding='utf-8'))
    run = json.loads(run_path.read_text(encoding='utf-8'))
    adapter = run['adapter']
    rows.append({'model': adapter['model_key'], 'model_id': adapter['model_id'], 'direction': adapter['direction'], 'dataset': metrics['dataset_role'], 'rows': metrics['rows'], 'chrF++': metrics['chrF++'], 'SacreBLEU': metrics['sacreBLEU'], 'revision': adapter['checkpoint_revision'], 'language_control': json.dumps(adapter['required_language_control'], ensure_ascii=False), 'generation_policy': adapter['generation_policy'], 'generation_overrides': json.dumps(adapter['generation_overrides'], ensure_ascii=False), 'generation_settings_source': adapter['generation_settings_source'], 'output_normalization': json.dumps(adapter['output_normalization'], ensure_ascii=False), 'input_length_policy': adapter['input_length_policy']['name'], 'metric_config': json.dumps(metrics['metric_config'], ensure_ascii=False), 'evidence': str(metric_path.relative_to(ROOT))})
table = pd.DataFrame(rows).sort_values(['direction','model','dataset']) if rows else pd.DataFrame()
comparison = table[['model', 'direction', 'dataset', 'rows', 'chrF++', 'SacreBLEU']]

# Mỗi bảng so sánh ba candidate trong cùng một chiều và cùng một validation set.
for direction, direction_label in [('en_to_vi', 'EN→VI'), ('vi_to_en', 'VI→EN')]:
    print(f'Kết quả {direction_label}')
    for dataset, dataset_label in [('selection_validation', 'IT Validation'), ('general_validation', 'General Validation')]:
        print(dataset_label)
        display(comparison.loc[(comparison['direction'] == direction) & (comparison['dataset'] == dataset), ['model', 'rows', 'chrF++', 'SacreBLEU']].reset_index(drop=True))

print('Provenance checkpoint — chỉ dùng để xác định model/checkpoint đã chạy:')
display(table[['model', 'direction', 'model_id', 'revision']].drop_duplicates().reset_index(drop=True))

print('Language control và cách xử lý input — chỉ dùng để kiểm tra cách chạy:')
display(table[['model', 'direction', 'language_control', 'input_length_policy']].drop_duplicates().reset_index(drop=True))

print('Runtime control và output normalization — chỉ dùng để kiểm tra cách gọi checkpoint:')
display(table[['model', 'direction', 'generation_policy', 'generation_overrides', 'generation_settings_source', 'output_normalization', 'metric_config']].drop_duplicates().reset_index(drop=True))

print('Đường dẫn evidence — chỉ dùng khi cần mở lại metrics.json hoặc run.json:')
table[['model', 'direction', 'dataset', 'evidence']]

Kết quả EN→VI
IT Validation


,model,rows,chrF++,SacreBLEU
0,envit5,15253,64.981633,48.974620
1,nllb200_distilled_600m,15253,60.368167,42.938858
2,opus_mt,15253,50.735648,31.937368


General Validation


,model,rows,chrF++,SacreBLEU
0,envit5,997,59.822968,38.398872
1,nllb200_distilled_600m,997,55.273235,33.588872
2,opus_mt,997,52.166180,29.697093


Kết quả VI→EN
IT Validation


...

,model,direction,language_control,input_length_policy
0,envit5,en_to_vi,"{""input_prefix"": ""en:"", ""input_prefix_role"": ""...",no_finite_checkpoint_input_limit
1,nllb200_distilled_600m,en_to_vi,"{""input_prefix"": """", ""input_prefix_role"": null...",deterministic_whitespace_then_character_segmen...
2,opus_mt,en_to_vi,"{""input_prefix"": "">>vie<<"", ""input_prefix_role...",deterministic_whitespace_then_character_segmen...
3,envit5,vi_to_en,"{""input_prefix"": ""vi:"", ""input_prefix_role"": ""...",no_finite_checkpoint_input_limit
4,nllb200_distilled_600m,vi_to_en,"{""input_prefix"": """", ""input_prefix_role"": null...",deterministic_whitespace_then_character_segmen...
5,opus_mt,vi_to_en,"{""input_prefix"": """", ""input_prefix_role"": ""tar...",deterministic_whitespace_then_character_segmen...


Runtime control và output normalization — chỉ dùng để kiểm tra cách gọi checkpoint:


,model,direction,generation_policy,generation_overrides,generation_settings_source,output_normalization,metric_config
0,envit5,en_to_vi,checkpoint_default_with_official_model_card_ru...,"{""max_length"": 512}",https://huggingface.co/VietAI/envit5-translation,"{""policy"": ""remove_exact_leading_control_prefi...","{""chrF++"": {""beta"": 2, ""word_order"": 2}, ""sacr..."
1,envit5,en_to_vi,checkpoint_default_with_official_model_card_ru...,"{""max_length"": 512}",https://huggingface.co/VietAI/envit5-translation,"{""policy"": ""remove_exact_leading_control_prefi...","{""chrF++"": {""beta"": 2, ""word_order"": 2}, ""sacr..."
2,nllb200_distilled_600m,en_to_vi,checkpoint_default,{},NaN,"{""policy"": ""none"", ""control_prefix"": null, ""no...","{""chrF++"": {""beta"": 2, ""word_order"": 2}, ""sacr..."
3,opus_mt,en_to_vi,checkpoint_default,{},NaN,"{""policy"": ""none"", ""control_prefix"": null, ""no...","{""chrF++"": {""beta"": 2, ""word_order"": 2}, ""sacr..."
4,envit5,vi_to_en,checkpoint_default_with_official_model_card_ru...,"{""max_length"": 512}",https://huggingface.co/VietAI/envit5-translation,"{""policy"": ""remove_exact_leading_control_prefi...","{""chrF++"": {""beta"": 2, ""word_order"": 2}, ""sacr..."
5,envit5,vi_to_en,checkpoint_default_with_official_model_card_ru...,"{""max_length"": 512}",https://huggingface.co/VietAI/envit5-translation,"{""policy"": ""remove_exact_leading_control_prefi...","{""chrF++"": {""beta"": 2, ""word_order"": 2}, ""sacr..."
6,nllb200_distilled_600m,vi_to_en,checkpoint_default,{},NaN,"{""policy"": ""none"", ""control_prefix"": null, ""no...","{""chrF++"": {""beta"": 2, ""word_order"": 2}, ""sacr..."
7,opus_mt,vi_to_en,checkpoint_default,{},NaN,"{""policy"": ""none"", ""control_prefix"": null, ""no...","{""chrF++"": {""beta"": 2, ""word_order"": 2}, ""sacr..."


Đường dẫn evidence — chỉ dùng khi cần mở lại metrics.json hoặc run.json:


,model,direction,dataset,evidence
0,envit5,en_to_vi,general_validation,evidence\phase02\envit5\en_to_vi\pretrained\ge...
1,envit5,en_to_vi,selection_validation,evidence\phase02\envit5\en_to_vi\pretrained\se...
4,nllb200_distilled_600m,en_to_vi,general_validation,evidence\phase02\nllb200_distilled_600m\en_to_...
5,nllb200_distilled_600m,en_to_vi,selection_validation,evidence\phase02\nllb200_distilled_600m\en_to_...
8,opus_mt,en_to_vi,general_validation,evidence\phase02\opus_mt\en_to_vi\pretrained\g...
9,opus_mt,en_to_vi,selection_validation,evidence\phase02\opus_mt\en_to_vi\pretrained\s...
2,envit5,vi_to_en,general_validation,evidence\phase02\envit5\vi_to_en\pretrained\ge...
3,envit5,vi_to_en,selection_validation,evidence\phase02\envit5\vi_to_en\pretrained\se...
6,nllb200_distilled_600m,vi_to_en,general_validation,evidence\phase02\nllb200_distilled_600m\vi_to_...
7,nllb200_distilled_600m,vi_to_en,selection_validation,evidence\phase02\nllb200_distilled_600m\vi_to_...


In [2]:
expected = {(m, d, s) for m in ('opus_mt','envit5','nllb200_distilled_600m') for d in ('en_to_vi','vi_to_en') for s in ('selection_validation','general_validation')}
actual = set(zip(table.model, table.direction, table.dataset)) if not table.empty else set()
print('Missing completed runs:', sorted(expected - actual))
print('Incomplete .running directories:', [str(p.relative_to(ROOT)) for p in BASE.glob('*/*/pretrained/*.running')])
assert not (expected - actual), 'Do not write selection record until all 12 completed runs exist.'

Missing completed runs: []
Incomplete .running directories: []


In [3]:
from hashlib import sha256

# Five row IDs per direction, selected only from row IDs by a fixed hash.
# This is an inspection aid, not a manual-review sample or a quality score.
for direction in ('en_to_vi', 'vi_to_en'):
    by_model = {}
    for model in ('opus_mt', 'envit5', 'nllb200_distilled_600m'):
        path = BASE / model / direction / 'pretrained' / 'selection_validation' / 'predictions.jsonl'
        by_model[model] = {row['row_id']: row for row in map(json.loads, path.read_text(encoding='utf-8').splitlines())}
    common_ids = set.intersection(*(set(rows) for rows in by_model.values()))
    sample_ids = sorted(common_ids, key=lambda row_id: sha256(f'phase02-screening:{direction}:{row_id}'.encode()).hexdigest())[:5]
    sample = []
    for row_id in sample_ids:
        first = by_model['opus_mt'][row_id]
        sample.append({'row_id': row_id, 'technical_tags': ', '.join(first['technical_tags']), 'source': first['source'], 'reference': first['reference'], **{model: by_model[model][row_id]['prediction'] for model in by_model}})
    print(direction)
    display(pd.DataFrame(sample))

en_to_vi


,row_id,technical_tags,source,reference,opus_mt,envit5,nllb200_distilled_600m
0,5579ad1b8d69575f1c45bb9270d401fe43d66734c5f3a7...,,Naughty Dog's behind-the-scenes documentary on...,Phim tài liệu hậu trường của Naughty Dog về Th...,Bộ phim tài liệu về The Last of us Part 2 đến ...,Phim tài liệu hậu trường của Dog Ngạo mạn về T...,Bộ phim tài liệu hậu trường của Naughty Dog về...
1,d6e9d700e3b6b200025a7904e287f3323da186438acaae...,has_error,From nursing to gaming: How a breastfeeding cu...,Từ cho con bú đến chơi game: Làm thế nào một c...,Từ y tá đến trò chơi: làm thế nào một cái đệm ...,Từ việc cho con bú đến chơi game: Làm thế nào ...,Từ y tá đến chơi game: Làm thế nào một chiếc g...
2,e61dc765916415f04d98c48aeece30b02304b7b7fe718f...,,Two Blades Horizontal Clockwise,Tắt theo chiều ngang,Tắt theo chiều ngang,Hai lưỡi ngang theo chiều kim đồng hồ,Hai thanh vây ngang theo chiều hướng đồng hồ
3,8135fd9ad97747fb591d63e7013837700efc9006bfa70e...,,Apple Watch ban temporarily paused,Lệnh cấm Apple Watch tạm thời bị tạm dừng,Comment,Tạm dừng xem Apple,Lệnh cấm Apple Watch tạm dừng
4,1e90e2acf5e10995aa59c5199b74afca02a9cf3ed3dcca...,,The race is on to commercialise fusion energy....,Cuộc đua nhằm thương mại hóa năng lượng nhiệt ...,Cuộc đua sẽ thương mại hóa năng lượng hợp hạch.,Cuộc đua để thương mại hóa năng lượng nhiệt hạ...,Cuộc đua về thương mại hóa năng lượng tổng hợp...


vi_to_en


,row_id,technical_tags,source,reference,opus_mt,envit5,nllb200_distilled_600m
0,0fc80732b33a309edb781cc2b2d2c8bfe90c50c67720d0...,,Video nhập vai của Apple giới thiệu các trận P...,Apple Immersive Video Featuring 2023 MLS Playo...,Apple's playing video introduces the Playoffs ...,Apple role - playing video showcasing the upco...,Apple's video of the 2023 MLS Playoffs is abou...
1,db7377cef5e4c2b538ea2fc85000e097edd4243dbceb60...,,Ứng dụng giao hàng trong 10 phút Zepto thu hút...,"Zepto, a 10-minute delivery app, raises $665M ...",The delivery application for 10 minutes of Zep...,The 10 - minute delivery application Zepto rai...,The 10-minute delivery app Zepto has attracted...
2,b17652697bf985b1db024135e309096c8c11d313e23af3...,,Kế,& Next,Next,Next,- What?
3,05713e970c01af6cdca545c678214941da1e2691ed04cb...,,Xe đạp điện Gogoro CrossOver với phạm vi chạy ...,Gogoro CrossOver electric bike with a 150km ra...,Gogoro CrossOver electric bicycles that run 15...,The Gogoro CrossOver electric bike with 150 km...,Gogoro CrossOver electric bike with a range of...
4,89f44448407d3969b0872d6339ee5d180df8c8eea61385...,,2 giây,2 secs,2 sec,2 sec,Two seconds.


Ghi `evidence/phase02/selection/en_to_vi.md` và `vi_to_en.md` bằng nhận định có lý do cụ thể. Dùng [selection_record.md](../../docs/phase_02/selection_record.md) làm mẫu; không chọn theo cell này.